Making Network1 html:

In [2]:
import pandas as pd

df = pd.read_csv("about_networks/genre_genre.csv")
print(df.columns)

Index(['Source', 'Target', 'Weight'], dtype='str')


In [20]:
from pyvis.network import Network
import pandas as pd
import networkx as nx

# ----------------------------
# 1. Load CSV
# ----------------------------
df = pd.read_csv("about_networks/genre_genre.csv")
df.columns = df.columns.str.strip()

# ----------------------------
# 2. Create NetworkX Graph
# ----------------------------
G = nx.from_pandas_edgelist(df, source="Source", target="Target", edge_attr="Weight")

# ----------------------------
# 3. Create PyVis Network
# ----------------------------
net = Network(height="800px", width="100%", bgcolor="#111111", font_color="white", notebook=False)
net.from_nx(G)

# ----------------------------
# 4. Physics + Hover Effects
# ----------------------------
net.set_options("""
{
  "nodes": {
    "borderWidth": 2,
    "size": 15,
    "color": {
      "border": "#ffffff",
      "background": "#1f78b4",
      "highlight": {
        "border": "#ffffff",
        "background": "#4db8ff"
      },
      "hover": {
        "border": "#ffffff",
        "background": "#4db8ff"
      }
    },
    "font": {
      "size": 14,
      "face": "Tahoma",
      "color": "#ffffff"
    },
    "physics": true
  },
  "edges": {
    "color": {
      "color": "#1f78b4",
      "highlight": "#03fc35"
    },
    "smooth": {
      "type": "continuous"
    }
  },
  "physics": {
    "enabled": true,
    "barnesHut": {
      "gravitationalConstant": -2500,
      "centralGravity": 0.15,
      "springLength": 250,
      "springConstant": 0.005,
      "damping": 0.35,
      "avoidOverlap": 0.5
    },
    "minVelocity": 0.02,
    "solver": "barnesHut",
    "stabilization": {
      "enabled": false
    }
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 100,
    "hoverConnectedEdges": true,
    "multiselect": true,
    "selectable": true
  }
}
""")

# ----------------------------
# 5. Export Interactive HTML
# ----------------------------
html_file = "genre_network.html"
net.write_html(html_file, notebook=False)

# ----------------------------
# 6. Inject custom JS for click highlight + title + info panel
# ----------------------------
custom_js = """
<script type="text/javascript">
(function() {
    var nodes = network.body.data.nodes;
    var edges = network.body.data.edges;

    // --- Click highlight ---
    network.on("selectNode", function(params) {
        for (var i = 0; i < params.nodes.length; i++) {
            var nodeId = params.nodes[i];
            var node = nodes.get(nodeId);
            node.color.background = "#03fc35"; // dark green on click
            nodes.update(node);
        }

        // --- Update info panel ---
        var nodeId = params.nodes[0];
        if (!nodeId) return;
        var connectedEdges = network.getConnectedEdges(nodeId);
        infoDiv.innerHTML = "<b>Node:</b> " + nodeId + " | <b>Edges:</b> " + connectedEdges.length;
    });

    network.on("deselectNode", function(params) {
        nodes.get().forEach(function(node) {
            if (!node.selected) {
                node.color.background = "#1f78b4"; // reset to original
                nodes.update(node);
            }
        });
        infoDiv.innerHTML = "";
    });

    // --- Info panel below canvas ---
    var infoDiv = document.createElement('div');
    infoDiv.style.position = 'absolute';
    infoDiv.style.bottom = '20px';
    infoDiv.style.left = '50%';
    infoDiv.style.transform = 'translateX(-50%)';
    infoDiv.style.color = '#ffffff';
    infoDiv.style.fontSize = '16px';
    infoDiv.style.fontFamily = 'Tahoma';
    infoDiv.style.backgroundColor = 'rgba(0,0,0,0.6)';
    infoDiv.style.padding = '8px 12px';
    infoDiv.style.borderRadius = '6px';
    infoDiv.style.pointerEvents = 'none';
    infoDiv.style.transition = 'all 0.3s ease-in-out';
    infoDiv.style.opacity = 0;
    document.body.appendChild(infoDiv);

    network.on("selectNode", function() { infoDiv.style.opacity = 1; });
    network.on("deselectNode", function() { infoDiv.style.opacity = 0; });

    // --- Network title above canvas ---
    var titleDiv = document.createElement('div');
    titleDiv.innerHTML = "Genre Network";
    titleDiv.style.position = 'absolute';
    titleDiv.style.top = '20px';  // top margin
    titleDiv.style.left = '50%';
    titleDiv.style.transform = 'translateX(-50%)';
    titleDiv.style.color = '#4db8ff';
    titleDiv.style.fontSize = '28px';
    titleDiv.style.fontWeight = 'bold';
    titleDiv.style.fontFamily = 'Tahoma';
    titleDiv.style.backgroundColor = 'transparent'; // no background, fully black behind
    titleDiv.style.border = 'none';  // remove any borders
    titleDiv.style.pointerEvents = 'none';
    document.body.appendChild(titleDiv);
})();
</script>
"""

# Append JS to HTML
with open(html_file, "r", encoding="utf-8") as f:
    html_content = f.read()

html_content = html_content.replace("</body>", custom_js + "\n</body>")

with open(html_file, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"Interactive network saved to {html_file} with clean black background and node edges info!")

Interactive network saved to genre_network.html with clean black background and node edges info!


Making Network2 html:

In [21]:
import pandas as pd

df = pd.read_csv("about_networks/anime_community_assignments.csv")
print(df.columns)

Index(['anime_id', 'anime_name', 'genres', 'num_genres',
       'assigned_communities', 'all_community_names', 'confidence',
       'is_bridge', 'communities_spanned', 'community_distribution'],
      dtype='str')


In [24]:
from pyvis.network import Network
import pandas as pd
import networkx as nx

# ----------------------------
# 1. Load CSV
# ----------------------------
df = pd.read_csv("about_networks/anime_community_assignments.csv")  # replace with your actual file
df.columns = df.columns.str.strip()

# ----------------------------
# 2. Create NetworkX Graph (fast version)
# ----------------------------
G = nx.Graph()
max_nodes_per_community = 50  # limit nodes per community for speed

for community in df['assigned_communities'].unique():
    members = df[df['assigned_communities'] == community]['anime_id'].tolist()[:max_nodes_per_community]
    for i in range(len(members)):
        for j in range(i+1, len(members)):
            G.add_edge(members[i], members[j])

# Add node labels
for _, row in df.iterrows():
    if row['anime_id'] in G.nodes:
        G.nodes[row['anime_id']]['label'] = row['anime_name']

# ----------------------------
# 3. Create PyVis Network
# ----------------------------
net = Network(height="800px", width="100%", bgcolor="#111111", font_color="white", notebook=False)
net.from_nx(G)

# ----------------------------
# 4. Physics + Hover Effects (distinct from Genre Network)
# ----------------------------
net.set_options("""
{
  "nodes": {
    "borderWidth": 2,
    "size": 16,
    "color": {
      "border": "#ffffff",
      "background": "#9b59b6",
      "highlight": {"border": "#ffffff","background": "#d29bff"},
      "hover": {"border": "#ffffff","background": "#d29bff"}
    },
    "font": {"size": 14,"face": "Tahoma","color": "#ffffff"},
    "physics": true
  },
  "edges": {
    "color": {"color": "#9b59b6","highlight": "#ff69b4"},
    "smooth": {"type": "continuous"}
  },
  "physics": {
    "enabled": true,
    "barnesHut": {
      "gravitationalConstant": -1800,
      "centralGravity": 0.25,
      "springLength": 200,
      "springConstant": 0.006,
      "damping": 0.4,
      "avoidOverlap": 0.6
    },
    "minVelocity": 0.01,
    "solver": "barnesHut",
    "stabilization": {"enabled": false}
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 100,
    "hoverConnectedEdges": true,
    "multiselect": true,
    "selectable": true
  }
}
""")
# ----------------------------
# 5. Export Interactive HTML
# ----------------------------
html_file = "anime_network.html"
net.write_html(html_file, notebook=False)

# ----------------------------
# 6. Inject JS for click highlight + title + info panel
# ----------------------------
custom_js = """
<script type="text/javascript">
(function() {
    var nodes = network.body.data.nodes;

    // Click highlight
    network.on("selectNode", function(params) {
        for (var i = 0; i < params.nodes.length; i++) {
            var nodeId = params.nodes[i];
            var node = nodes.get(nodeId);
            node.color.background = "#ff69b4"; // pink on click
            nodes.update(node);
        }
        var nodeId = params.nodes[0];
        if (!nodeId) return;
        var connectedEdges = network.getConnectedEdges(nodeId);
        infoDiv.innerHTML = "<b>Node:</b> " + nodes.get(nodeId).label + " | <b>Edges:</b> " + connectedEdges.length;
    });

    network.on("deselectNode", function(params) {
        nodes.get().forEach(function(node) {
            if (!node.selected) {
                node.color.background = "#9b59b6";
                nodes.update(node);
            }
        });
        infoDiv.innerHTML = "";
    });

    // Info panel below canvas
    var infoDiv = document.createElement('div');
    infoDiv.style.position = 'absolute';
    infoDiv.style.bottom = '20px';
    infoDiv.style.left = '50%';
    infoDiv.style.transform = 'translateX(-50%)';
    infoDiv.style.color = '#ffffff';
    infoDiv.style.fontSize = '16px';
    infoDiv.style.fontFamily = 'Tahoma';
    infoDiv.style.backgroundColor = 'rgba(0,0,0,0.6)';
    infoDiv.style.padding = '8px 12px';
    infoDiv.style.borderRadius = '6px';
    infoDiv.style.pointerEvents = 'none';
    infoDiv.style.transition = 'all 0.3s ease-in-out';
    infoDiv.style.opacity = 0;
    document.body.appendChild(infoDiv);

    network.on("selectNode", function() { infoDiv.style.opacity = 1; });
    network.on("deselectNode", function() { infoDiv.style.opacity = 0; });

    // Network title above canvas
    var titleDiv = document.createElement('div');
    titleDiv.innerHTML = "Anime Network";
    titleDiv.style.position = 'absolute';
    titleDiv.style.top = '30px';  // slightly more top margin
    titleDiv.style.left = '50%';
    titleDiv.style.transform = 'translateX(-50%)';
    titleDiv.style.color = '#d29bff';
    titleDiv.style.fontSize = '28px';
    titleDiv.style.fontWeight = 'bold';
    titleDiv.style.fontFamily = 'Tahoma';
    titleDiv.style.backgroundColor = 'transparent';
    titleDiv.style.border = 'none';
    titleDiv.style.pointerEvents = 'none';
    document.body.appendChild(titleDiv);
})();
</script>
"""

with open(html_file, "r", encoding="utf-8") as f:
    html_content = f.read()

html_content = html_content.replace("</body>", custom_js + "\n</body>")

with open(html_file, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"Fast Anime Network saved to {html_file} with distinct physics and purple theme!")

Fast Anime Network saved to anime_network.html with distinct physics and purple theme!
